# Runtime (Polyglot)

In [1]:
#r "nuget: fsharp.control.asyncseq, 3.2.1"
#r "nuget: system.reactive, 6.0.1-preview.1"
#r "nuget: system.reactive.linq, 6.0.1-preview.1"
#r "nuget: argu, 6.2.4"

In [2]:
#!import ../../lib/fsharp/Notebooks.livemd

In [3]:
#!import ../../lib/fsharp/Testing.livemd

In [4]:
#!import ../../lib/fsharp/Common.fs

In [5]:
#!import ../../lib/fsharp/CommonFSharp.fs

In [6]:
#!import ../../lib/fsharp/Async.fs

In [7]:
#!import ../../lib/fsharp/AsyncSeq.fs

In [8]:
#!import ../../lib/fsharp/Runtime.fs

In [9]:
#!import ../../lib/fsharp/FileSystem.fs

In [10]:
#if !INTERACTIVE
open Lib
#endif

In [11]:
open Common

In [12]:
///- --test

open SpiralFileSystem.Operators

## parseArgs

In [13]:
let inline parseArgs<'T when 'T :> Argu.IArgParserTemplate> args =
    let assemblyName = System.Reflection.Assembly.GetEntryAssembly().GetName().Name
    let runsInNotebookCell = assemblyName = "fsi"
    let errorHandler : Argu.IExiter =
        if runsInNotebookCell
        then Argu.ExceptionExiter ()
        else Argu.ProcessExiter (function Argu.ErrorCode.HelpText -> None | _ -> Some System.ConsoleColor.Red)

    let parser =
        Argu.ArgumentParser.Create<'T> (
            programName = $"{assemblyName}{SpiralPlatform.get_executable_suffix ()}",
            errorHandler = errorHandler
        )

    parser.ParseCommandLine args

In [14]:
///- --test

[<RequireQualifiedAccess>]
type Arguments =
    | [<Argu.ArguAttributes.MainCommand; Argu.ArguAttributes.ExactlyOnce; Argu.ArguAttributes.Last>]
        Paths of paths : string list

    interface Argu.IArgParserTemplate with
        member s.Usage =
            match s with
            | Paths _ -> nameof Paths

In [15]:
///- --test

Argu.ArgumentParser.Create<Arguments>().PrintUsage ()

"USAGE: fsi [--help] <paths>...

PATHS:

    <paths>...            Paths

OPTIONS:

    --help                display this list of options.
"

In [16]:
///- --test

fun () -> parseArgs<Arguments> [||] |> ignore
|> _throwsC (fun ex _ ->
    SpiralSm.format_exception ex
    |> _stringContains "Argu.ArguParseException: ERROR: missing parameter '<paths>...'."
)

<fun:it@49270-1>

"Argu.ArguParseException: ERROR: missing parameter '<paths>...'.
USAGE: fsi [--help] <paths>...

PATHS:

    <paths>...            Paths

OPTIONS:

    --help                display this list of options.
"

In [17]:
let inline parseAllArgs<'T when 'T :> Argu.IArgParserTemplate> args =
    args
    |> parseArgs<'T>
    |> fun results -> results.GetAllResults ()

In [18]:
///- --test

[<RequireQualifiedAccess>]
type Arguments =
    | [<Argu.ArguAttributes.MainCommand; Argu.ArguAttributes.ExactlyOnce; Argu.ArguAttributes.Last>]
        Paths of paths : string list

    interface Argu.IArgParserTemplate with
        member s.Usage =
            match s with
            | Paths _ -> nameof Paths

parseAllArgs<Arguments> [| "a b"; "c" |]
|> _assertEqual [ Arguments.Paths [ "a b"; "c" ] ]

[Paths ["a b"; "c"]]

In [19]:
let inline parseArgsMap<'T when 'T :> Argu.IArgParserTemplate> args =
    args
    |> parseAllArgs<'T>
    |> List.groupBy CommonFSharp.getUnionCaseName<'T>
    |> Map.ofList

In [20]:
///- --test

parseArgsMap<Arguments> [| "a b"; "c" |]
|> _assertEqual (
    [ nameof Arguments.Paths, [ Arguments.Paths [ "a b"; "c" ] ] ]
    |> Map.ofList
)



/home/runner/work/polyglot/polyglot/lib/fsharp/input.fsx(5,5): info FS3886: This list expression contains a single tuple element. Did you mean to use ';' instead of ',' to separate list elements?

map [("Paths", [Paths ["a b"; "c"]])]